In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from config.settings import DATA_DIR, OUTPUT_DIR
from src.utils.data_store import load
from src.analysis.dtm import build_dtm
from src.cleaning.text_cleaner import clean_scopus
from src.visualization.wordcloud_viz import plot_wordcloud, plot_comparison_cloud
from src.visualization.dendrogram import plot_dendrogram

df = load("scopus_devops", DATA_DIR)

# Use Abstract_clean when available; fall back to Title_clean or cleaned Title
abstract_texts = [t for t in df["Abstract_clean"].dropna().tolist() if str(t).strip()]
if len(abstract_texts) < len(df) * 0.1:
    if "Title_clean" in df.columns:
        title_texts = [t for t in df["Title_clean"].dropna().tolist() if str(t).strip()]
    else:
        title_texts = [clean_scopus(str(t)) for t in df["Title"].dropna().tolist() if str(t).strip()]
    texts = abstract_texts + title_texts
    print(f"Only {len(abstract_texts)}/{len(df)} abstracts — using titles as fallback")
else:
    texts = abstract_texts
texts = [t for t in texts if t.strip()]
print(f"Loaded {len(texts)} documents")

In [ ]:
dtm, vectorizer = build_dtm(texts)
print(f"DTM shape: {dtm.shape}")
vocab = vectorizer.get_feature_names_out()
print("Top 20 terms:", list(vocab[:20]))

In [ ]:
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
path = plot_wordcloud(texts, output_path=f"{OUTPUT_DIR}/wordcloud_scopus.png")
from IPython.display import Image; Image(path)

In [ ]:
df["Year"] = pd.to_datetime(df["Date"], errors="coerce").dt.year
early = df[df["Year"] < 2020]["Abstract_clean"].dropna().tolist()
recent = df[df["Year"] >= 2020]["Abstract_clean"].dropna().tolist()
if early and recent:
    path = plot_comparison_cloud({"Before 2020": early, "2020+": recent}, output_path=f"{OUTPUT_DIR}/comparison_cloud.png")
    from IPython.display import Image; Image(path)
else:
    print("Not enough data in both periods for comparison cloud")

In [ ]:
path = plot_dendrogram(dtm, output_path=f"{OUTPUT_DIR}/dendrogram_scopus.pdf")
print(f"Dendrogram saved to {path}")